<a href="https://colab.research.google.com/github/bandi137/NASA_Nature_Event_Monitor/blob/main/NASA_Nature_Event_Monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import folium
from folium.plugins import MarkerCluster

# 1. Adatok lekérése - Ha a days=None, minden nyitottat lekér
def get_eonet_events(days=None):
    if days:
        url = f"https://eonet.gsfc.nasa.gov/api/v3/events?status=open&days={days}"
    else:
        url = "https://eonet.gsfc.nasa.gov/api/v3/events?status=open"

    headers = {'User-Agent': 'Mozilla/5.0 (Colab; NASA_EONET_App)'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()['events']
        print(f"DEBUG: Az API összesen {len(data)} nyitott eseményt küldött vissza globálisan.")
        return data
    except Exception as e:
        print(f"Hiba az adatok lekérésekor: {e}")
        return []

# 2. Szűrési függvények
def is_in_europe(lat, lon):
    return 35 <= lat <= 70 and -25 <= lon <= 45

def is_in_carpathian_basin(lat, lon):
    return 44 <= lat <= 49 and 16 <= lon <= 27

def get_event_style(category_id):
    styles = {
        'wildfires': {'color': 'red', 'icon': 'fire'},
        'seaLakeIce': {'color': 'blue', 'icon': 'snowflake'},
        'volcanoes': {'color': 'darkred', 'icon': 'info-sign'},
        'severeStorms': {'color': 'gray', 'icon': 'cloud'},
        'dustHaze': {'color': 'orange', 'icon': 'eye-open'},
        'waterColor': {'color': 'cadetblue', 'icon': 'tint'}
    }
    return styles.get(category_id, {'color': 'green', 'icon': 'question-sign'})

# Fő folyamat - Visszaállítjuk None-ra, hogy lássunk mindent, ami nyitott
events = get_eonet_events(days=None)

m = folium.Map(location=[50, 20], zoom_start=4, tiles='CartoDB Voyager')
european_events_count = 0
carpathian_events = []

for event in events:
    if 'geometry' in event and event['geometry']:
        geo = event['geometry'][-1]
        if geo['type'] == 'Point':
            lon, lat = geo['coordinates']
            if is_in_europe(lat, lon):
                european_events_count += 1
                style = get_event_style(event['categories'][0]['id'])
                is_carpathian = is_in_carpathian_basin(lat, lon)
                color = 'purple' if is_carpathian else style['color']
                if is_carpathian: carpathian_events.append(event['title'])

                source_link = event['sources'][0]['url'] if event['sources'] else "Nincs"
                popup_text = f"<b>{event['title']}</b><br>{event['categories'][0]['title']}<br><a href='{source_link}'>Forrás</a>"

                folium.Marker(
                    location=[lat, lon], popup=folium.Popup(popup_text, max_width=300),
                    icon=folium.Icon(color=color, icon=style['icon'])
                ).add_to(m)

display(m)
print(f"\n--- ÖSSZEFOGLALÓ ---\nEurópai találatok: {european_events_count}")

DEBUG: Az API összesen 6543 nyitott eseményt küldött vissza globálisan.



--- ÖSSZEFOGLALÓ ---
Európai találatok: 28
